# Frontière efficiente de Markowitz – 2025
Objectif : construire la frontière efficiente d'un portefeuille de 4 actions (Thales, TotalEnergies, Tesla, LVMH) sur l'année 2025 et identifier le portefeuille au ratio de Sharpe maximal.

Méthode : rendements logarithmiques journaliers, simulation de 10 000 portefeuilles aux poids aléatoires, puis optimisation sous contraintes (SciPy, SLSQP) : poids positifs, somme égale à 1.

In [ ]:
import datetime
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

## 1. Données

In [ ]:
start = datetime.datetime(2025,1,1)
end = datetime.datetime(2025,12,31)

thales = yf.download('HO.PA', start=start, end=end, auto_adjust=True, progress=False)
total  = yf.download('TTE.PA', start=start, end=end, auto_adjust=True, progress=False)
tesla  = yf.download('TSLA',  start=start, end=end, auto_adjust=True, progress=False)
lvmh   = yf.download('MC.PA', start=start, end=end, auto_adjust=True, progress=False)

stocks = pd.concat([thales['Close'], total['Close'], tesla['Close'], lvmh['Close']], axis=1, sort=True)
stocks.columns = ['THALES', 'TOTALEN', 'TESLA', 'LVMH']
stocks = stocks.dropna()   # garde seulement les jours où Paris ET New York sont ouverts
stocks

## 2. Rendements logarithmiques
$r_t = \ln(p_t / p_{t-1})$

In [ ]:
logReturns = np.log(stocks / stocks.shift(1)).dropna()
logReturns

## 3. Paramètres
Taux sans risque : hypothèse €STR ≈ 2 % par an, converti en taux journalier pour être cohérent avec les rendements.

In [ ]:
rf_annual = 0.02
rf = rf_annual / 252
meanLogReturns = logReturns.mean()
Sigma = logReturns.cov()

## 4. Simulation de 10 000 portefeuilles aléatoires

In [ ]:
noOfPortfolios = 10000
weight = np.zeros((noOfPortfolios, 4))
expectedReturn = np.zeros(noOfPortfolios)
expectedVolatility = np.zeros(noOfPortfolios)
sharpeRatio = np.zeros(noOfPortfolios)

for k in range(noOfPortfolios):
    w = np.random.random(4)
    w = w / np.sum(w)
    weight[k, :] = w
    expectedReturn[k] = np.sum(meanLogReturns * w)
    expectedVolatility[k] = np.sqrt(w.T @ Sigma @ w)
    sharpeRatio[k] = (expectedReturn[k] - rf) / expectedVolatility[k]

maxIndex = sharpeRatio.argmax()
weight[maxIndex, :]

## 5. Optimisation : portefeuille au Sharpe maximal
Comparaison pour trois niveaux de taux sans risque (0 %, 2 %, 4 %). Résultats annualisés : rendement × 252, volatilité × √252.

In [ ]:
def getReturn(w):
    return np.sum(meanLogReturns * np.array(w))

def getVol(w):
    w = np.array(w)
    return np.sqrt(w.T @ Sigma @ w)

def negativeSR(w):
    return -(getReturn(w) - rf) / getVol(w)

w0 = [0.25] * 4
bounds = [(0, 1)] * 4
sumToOne = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

for rf_test in [0, 0.02, 0.04]:
    rf = rf_test / 252
    w = minimize(negativeSR, w0, method='SLSQP', bounds=bounds, constraints=[sumToOne]).x
    R_ann = getReturn(w) * 252
    vol_ann = getVol(w) * np.sqrt(252)
    print(f"rf = {rf_test:.0%} | poids : {dict(zip(stocks.columns, [round(float(x), 3) for x in w]))}")
    print(f"   rendement {R_ann:.1%} | vol {vol_ann:.1%} | Sharpe annualisé {(R_ann - rf_test) / vol_ann:.2f}")

rf = rf_annual / 252   # on revient à l'hypothèse de base
w_opt = minimize(negativeSR, w0, method='SLSQP', bounds=bounds, constraints=[sumToOne]).x
print("Poids optimaux (rf = 2 %) :", dict(zip(stocks.columns, [round(float(x), 3) for x in w_opt])))

## 6. Frontière efficiente

In [ ]:
targetReturns = np.linspace(meanLogReturns.min(), meanLogReturns.max(), 50)
frontierVol = []
for R in targetReturns:
    cons = [sumToOne, {'type': 'eq', 'fun': lambda w, R=R: getReturn(w) - R}]
    frontierVol.append(minimize(getVol, w0, method='SLSQP', bounds=bounds, constraints=cons).fun)
frontierVol = np.array(frontierVol)

plt.figure(figsize=(12, 6))
plt.scatter(expectedVolatility * np.sqrt(252), expectedReturn * 252, c=sharpeRatio * np.sqrt(252), s=5)
plt.colorbar(label='Sharpe annualisé')
plt.plot(frontierVol * np.sqrt(252), targetReturns * 252, 'k-', lw=2, label='Frontière efficiente')
plt.scatter(getVol(w_opt) * np.sqrt(252), getReturn(w_opt) * 252, c='red', s=80, label='Sharpe max')
plt.xlabel('Volatilité annualisée')
plt.ylabel('Rendement log annualisé')
plt.title('Frontière efficiente de Markowitz (2025)')
plt.legend()
plt.show()

## 7. Limites
- Poids *backward-looking* : les rendements moyens de 2025 servent de rendements attendus, alors qu'estimés sur un an ils sont très imprécis.
- L'optimiseur amplifie ces erreurs d'estimation (forte sensibilité des poids aux moyennes).
- Tesla est cotée en USD : le rendement pour un investisseur en euros inclut l'effet de change EUR/USD.
- Pistes : test hors échantillon (estimation 2024, test 2025), fenêtre plus longue, plafonds de poids, portefeuille de variance minimale.